In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

import numpy as np
import seaborn as sns
from sklearn.metrics import f1_score, confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)

In [ ]:
df = pd.read_csv("data/complaints.csv", nrows=1000)
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
labels = pd.read_csv("data/complaints.csv", usecols=["Product"])
print(len(labels))
labels["Product"].value_counts().to_frame()

In [ ]:
df = pd.read_csv("data/complaints.csv")
print(df.isna().sum().to_frame())

In [ ]:
df["Product"].value_counts().plot(kind="barh", figsize=(10, 8))
plt.title("Nombre de demandes par catégorie")
plt.tight_layout()
plt.show()

In [ ]:
longueurs = df["Consumer complaint narrative"].str.len()
print(longueurs.describe())
print("Doublons :", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
df["texte"] = (
    df["Consumer complaint narrative"]
    .str.replace(r"X{2,}", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(len(df))
df[["Product", "texte"]].head()

In [ ]:
df["Product"].value_counts().to_frame()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["texte"],
    df["Product"],
    test_size=0.2,
    stratify=df["Product"],
    random_state=42,
)

print(y_train.nunique(), y_test.nunique())
pd.concat([y_train.value_counts(normalize=True).rename("train"), y_test.value_counts(normalize=True).rename("test")], axis=1)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=5,
    sublinear_tf=True,
    stop_words="english",
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

In [ ]:
X_train_tfidf = X_train_tfidf.astype("float32")
X_test_tfidf = X_test_tfidf.astype("float32")

In [ ]:
encoder = LabelEncoder()
y_train_enc = encoder.fit_transform(y_train)
y_test_enc = encoder.transform(y_test)

print(len(encoder.classes_))

In [ ]:
def create_model(n_classes):
    model = tf.keras.Sequential([
        layers.Input(shape=(X_train_tfidf.shape[1],)),
        layers.Dense(256, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["sparse_categorical_accuracy"],
    )
    return model

create_model(len(encoder.classes_)).summary()


In [ ]:
early = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

model = create_model(len(encoder.classes_))

history = model.fit(
    X_train_tfidf,
    y_train_enc,
    validation_split=0.1,
    epochs=30,
    batch_size=256,
    callbacks=[early],
)

In [ ]:
plt.plot(history.history["loss"], label="entraînement")
plt.plot(history.history["val_loss"], label="validation")
plt.xticks(range(len(history.history["loss"])))
plt.xlabel("epoch")
plt.ylabel("perte")
plt.legend()
plt.show()

In [ ]:
y_pred = np.argmax(model.predict(X_test_tfidf), axis=1)

print("Weighted F1 :", f1_score(y_test_enc, y_pred, average="weighted", zero_division=0))
print(classification_report(y_test_enc, y_pred, labels=range(21), target_names=encoder.classes_, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test_enc, y_pred, labels=range(21), normalize="true")

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt=".0%", cmap="Blues", cbar=False, xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Prédictions $\\hat{y}$")
plt.ylabel("Réponses $y$")
plt.tight_layout()
plt.show()

In [ ]:
FUSION = {
    # Credit reporting — 3 libellés, 55 % des données
    "Credit reporting": "Credit reporting",
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    # Cartes — 3 libellés
    "Credit card": "Credit card or prepaid card",
    "Credit card or prepaid card": "Credit card or prepaid card",
    "Prepaid card": "Credit card or prepaid card",
    # Compte bancaire — ancien / nouveau nom
    "Bank account or service": "Checking or savings account",
    "Checking or savings account": "Checking or savings account",
    # Transfert d'argent — 3 libellés
    "Money transfer, virtual currency, or money service": "Money transfer or virtual currency",
    "Money transfers": "Money transfer or virtual currency",
    "Virtual currency": "Money transfer or virtual currency",
    # Crédit à la consommation — 3 libellés
    "Payday loan": "Payday or personal loan",
    "Payday loan, title loan, or personal loan": "Payday or personal loan",
    "Payday loan, title loan, personal loan, or advance loan": "Payday or personal loan",
    # Auto / consommation — tranché par la matrice de confusion
    "Consumer Loan": "Vehicle or consumer loan",
    "Vehicle loan or lease": "Vehicle or consumer loan",
    # Catégories autonomes, inchangées
    "Debt collection": "Debt collection",
    "Mortgage": "Mortgage",
    "Student loan": "Student loan",
    # Résidus rares, laissés seuls : ce ne sont les doublons de personne
    "Debt or credit management": "Debt or credit management",
    "Other financial service": "Other financial service",
  }

y_train_f = y_train.map(FUSION)
y_test_f = y_test.map(FUSION)

assert y_train_f.notna().all() and y_test_f.notna().all(), "libellé absent du mapping"
print(y_train_f.nunique(), y_test_f.nunique())

In [ ]:
encoder_f = LabelEncoder()
y_train_f_enc = encoder_f.fit_transform(y_train_f)
y_test_f_enc = encoder_f.transform(y_test_f)

print(len(encoder_f.classes_))

In [ ]:
early_f = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

model_f = create_model(len(encoder_f.classes_))

history_f = model_f.fit(
    X_train_tfidf,
    y_train_f_enc,
    validation_split=0.1,
    epochs=30,
    batch_size=256,
    callbacks=[early_f],
)

In [ ]:
plt.plot(history_f.history["loss"], label="entraînement")
plt.plot(history_f.history["val_loss"], label="validation")
plt.xticks(range(len(history_f.history["loss"])))
plt.xlabel("epoch")
plt.ylabel("perte")
plt.legend()
plt.show()

In [ ]:
y_pred_f = np.argmax(model_f.predict(X_test_tfidf), axis=1)

print("Weighted F1 :", f1_score(y_test_f_enc, y_pred_f, average="weighted", zero_division=0))
print(classification_report(y_test_f_enc, y_pred_f,
                            target_names=encoder_f.classes_, zero_division=0))

In [ ]:
cm_f = confusion_matrix(y_test_f_enc, y_pred_f, normalize="true")

plt.figure(figsize=(10, 8))
sns.heatmap(cm_f, annot=True, fmt=".0%", cmap="Blues", cbar=False, xticklabels=encoder_f.classes_, yticklabels=encoder_f.classes_)
plt.xlabel("Prédictions $\\hat{y}$")
plt.ylabel("Réponses $y$")
plt.tight_layout()
plt.show()